<h1 style="color: #f686bd">Aprendizado por Reforço: Q-Learning do Zero</h1>

Chegamos ao último tipo de Aprendizado de Máquina! 

O **Aprendizado por Reforço (RL)** é fundamentalmente diferente: **ele não usa base de dados históricas (arquivos CSV)**. Ao invés disso, nós soltamos um **Agente** (nossa Inteligência Artificial) dentro de um **Ambiente** (um jogo, um labirinto, o mercado financeiro) e deixamos ele aprender por *tentativa e erro*.

Aqui, não usaremos o Scikit-Learn. Vamos programar a inteligência mais clássica de RL — o **Q-Learning** — usando Python e Matemática básica (NumPy) para fazer um "Robô" aprender a caminhar em um tabuleiro virtual 3x3 sem cair no buraco.

### Os Elementos do Jogo
- 🤖 **Agente:** Quem toma as decisões (Nosso Robô).
- 🎮 **Ambiente:** Onde o agente está (Nosso grid 3x3).
- 📍 **Estado:** Onde o agente está *agora* (Ex: Posição [0,0]).
- 🕹️ **Ação:** O que ele pode fazer (Ir para Cima, Baixo, Esquerda, Direita).
- 🏆 **Recompensa:** Pontuação que ele ganha ou perde ao dar um passo.

In [1]:
import numpy as np
import random
import time
from IPython.display import clear_output

np.set_printoptions(precision=2, suppress=True)

<h3 style="color: #f686bd">1. Criando o Ambiente (O Labirinto 3x3)</h3>
Vamos imaginar um pequeno mapa assim:
```text
[ INÍCIO ] [ Caminho] [ Caminho ]
[ Caminho] [ BURACO ] [ Caminho ]
[ Caminho] [ Caminho] [ DESTINO ]
```
As recompensas dão sentido à vida do robô:
- Cair no buraco: **-100 pontos** (Fim de jogo)
- Chegar no Destino: **+100 pontos** (Fim de jogo e Vitória)
- Dar um passo em falso (bater na parede ou andar à toa): **-1 ponto** (Para incentivar o robô a ser rápido)

In [2]:
# O mapa tem 9 posições (estados de 0 a 8)
NUM_ESTADOS = 9  

# O robô tem 4 ações possíveis (0=Cima, 1=Baixo, 2=Esquerda, 3=Direita)
NUM_ACOES = 4 

# Posições importantes
ESTADO_INICIAL = 0
BURACO = 4
DESTINO = 8

print(f"Temos {NUM_ESTADOS} casas e o robô tem {NUM_ACOES} botões no controle.")

Temos 9 casas e o robô tem 4 botões no controle.


<h3 style="color: #f686bd">A Inteligência do Robô: A Q-Table</h3>
Como o robô guarda memória? Através de uma tabela chamada **Q-Table**. 

A Q-Table é literalemente uma planilha Excel na "cabeça" dele. Cada **linha é uma casa do mapa** (Estado) e cada **coluna é uma ação** (Cima, Baixo, Esquerda, Direita). O valor dentro da célula diz para ele: *"O quão bom é dar este passo estando nesta casa?"*.
No começo, o robô não sabe nada. A tabela é preenchida com ZEROS.

In [3]:
# Criando a Q-Table vazia (Tudo Zero!)
q_table = np.zeros((NUM_ESTADOS, NUM_ACOES))

print("Memória Inicial do Robô (Q-Table):")
print(q_table)

Memória Inicial do Robô (Q-Table):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


<h3 style="color: #f686bd">2. As Regras da Física (Lógica do Movimento)</h3>
Precisamos programar o que acontece no nosso mundo quando o robô aperta um botão.

In [4]:
def dar_passo(estado_atual, acao):
    # Estado atual é um número de 0 a 8.
    # Imaginando o Grid:
    # 0  1  2
    # 3  4  5
    # 6  7  8
    
    novo_estado = estado_atual
    
    # Lógica de movimentação (com paredes invisíveis nas bordas!)
    if acao == 0 and estado_atual > 2:       novo_estado -= 3  # Cima
    elif acao == 1 and estado_atual < 6:     novo_estado += 3  # Baixo
    elif acao == 2 and estado_atual % 3 > 0: novo_estado -= 1  # Esquerda
    elif acao == 3 and estado_atual % 3 < 2: novo_estado += 1  # Direita
        
    # Calculando a Recompensa e verificando se o jogo acabou (Game Over)
    if novo_estado == DESTINO:
        return novo_estado, 100, True # Recompensa +100 e Vitória
    elif novo_estado == BURACO:
        return novo_estado, -100, True # Recompensa -100 e Morte
    else:
        # Perde 1 ponto de energia a cada passo pra não ficar rodando em círculos
        return novo_estado, -1, False 


<hr>
<h3 style="color: #f686bd">3. O Treinamento (A Magia do Q-Learning)</h3>
Chegou a hora de soltar o robô. Ele vai jogar esse jogo 5.000 vezes seguidas (Episódios).

No começo, ele explora clicando botões aleatoriamente. Depois, conforme preenche sua Q-Table com as experiências, ele começa a tomar decisões baseadas no que aprendeu (taxa de aprendizado ou exploração).

In [5]:
EPISODIOS = 5000
TAXA_APRENDIZADO = 0.1 # Alpha: Quanta atenção ele dá pra informação nova vs velha?
FATOR_DESCONTO = 0.9   # Gamma: Ele prefere recompensas agora ou pensa no futuro?
CHANCE_EXPLORACAO = 1.0 # Epsilon: Probabilidade inicial de clicar botões aleatórios (100%)
DECAIMENTO = 0.01      # A taxa que a exploração cai com o tempo

# Resetando a cabeça do robô para começar
q_table = np.zeros((NUM_ESTADOS, NUM_ACOES))

for episodio in range(EPISODIOS):
    estado = ESTADO_INICIAL
    game_over = False
    
    while not game_over:
        # 1. ESCOLHER AÇÃO (Explorar aleatório ou usar a Q-Table?)
        if random.uniform(0, 1) < CHANCE_EXPLORACAO:
            acao = random.randint(0, 3) # Botão Aleatório (Teste)
        else:
            acao = np.argmax(q_table[estado]) # Pega a melhor ação que ele já conhece pra esse estado
            
        # 2. EXECUTAR AÇÃO no mundo
        proximo_estado, recompensa, game_over = dar_passo(estado, acao)
        
        # 3. APRENDER (A Fórmula de Algoritmo de Bellman!)
        q_antigo = q_table[estado, acao]
        max_q_futuro = np.max(q_table[proximo_estado]) # Qual a pontuação máxima possível no próximo passo?
        
        # Atualizando a memória: Valor Velho + Aprendizado * (Recompensa Imediata + Recompensa Futura - Valor Velho)
        q_novo = q_antigo + TAXA_APRENDIZADO * (recompensa + FATOR_DESCONTO * max_q_futuro - q_antigo)
        q_table[estado, acao] = q_novo
        
        # 4. ANDAR (O vizual agora é a realidade)
        estado = proximo_estado
        
    # Decrementa a chance de botões aleatórios conforme ele fica inteligente
    CHANCE_EXPLORACAO = max(0.01, CHANCE_EXPLORACAO - DECAIMENTO)

print("TREINAMENTO CONCLUÍDO! O Robô jogou 5.000 vezes.")
print("\nQ-Table Final (A Inteligência Adquirida):")
print(np.round(q_table, 1))

TREINAMENTO CONCLUÍDO! O Robô jogou 5.000 vezes.

Q-Table Final (A Inteligência Adquirida):
[[ 42.7  25.4  49.7  70.2]
 [ 49.6 -96.6  42.7  79.1]
 [ 68.5  89.   49.4  63.9]
 [ 49.9   0.2  -0.8 -57. ]
 [  0.    0.    0.    0. ]
 [ 58.4 100.  -90.2  65.1]
 [ -0.2  -0.2  -0.3   5.5]
 [-27.1   0.8   0.   27.1]
 [  0.    0.    0.    0. ]]


<h3 style="color: #f686bd">4. Testando o Robô (O Resultado Visual)</h3>
Agora que a tabela está cheia de números inteligentes (valores positivos apontam para o destino e negativos desviam do buraco), vamos pedir para o Robô jogar o jogo mais uma vez e imprimir o mapa na tela, sempre escolhendo a melhor ação da sua tabela `q_table`.

In [6]:
def desenhar_mapa(estado_robo):
    mapa = ["-"] * 9
    mapa[DESTINO] = "🏅"
    mapa[BURACO] = "🕳️"
    mapa[estado_robo] = "🤖" # Substitui o que tiver pelo robô
    
    print(f"{mapa[0]:<4} {mapa[1]:<4} {mapa[2]:<4}")
    print(f"{mapa[3]:<4} {mapa[4]:<4} {mapa[5]:<4}")
    print(f"{mapa[6]:<4} {mapa[7]:<4} {mapa[8]:<4}")
    print("-----------------")

acoes_nome = ["Cima", "Baixo", "Esquerda", "Direita"]
estado = ESTADO_INICIAL
game_over = False
passos = 0

print("🎬 TESTE FINAL DO ROBOZINHO\n")

while not game_over:
    desenhar_mapa(estado)
    
    # Pega a melhor AÇÃO aprendida na tabela
    melhor_acao = np.argmax(q_table[estado])
    print(f"> O robô decidiu ir para: {acoes_nome[melhor_acao]}\n")
    
    estado, recompensa, game_over = dar_passo(estado, melhor_acao)
    passos += 1

desenhar_mapa(estado)
if estado == DESTINO:
    print(f"🎉 VITÓRIA! O robô aprendeu sozinho a desviar do buraco e chegar em {passos} passos!")
else:
    print("💀 GAME OVER: Ele caiu no buraco. Algo deu errado no treino.")

🎬 TESTE FINAL DO ROBOZINHO

🤖    -    -   
-    🕳️   -   
-    -    🏅   
-----------------
> O robô decidiu ir para: Direita

-    🤖    -   
-    🕳️   -   
-    -    🏅   
-----------------
> O robô decidiu ir para: Direita

-    -    🤖   
-    🕳️   -   
-    -    🏅   
-----------------
> O robô decidiu ir para: Baixo

-    -    -   
-    🕳️   🤖   
-    -    🏅   
-----------------
> O robô decidiu ir para: Baixo

-    -    -   
-    🕳️   -   
-    -    🤖   
-----------------
🎉 VITÓRIA! O robô aprendeu sozinho a desviar do buraco e chegar em 4 passos!


✅ **O que acabamos de fazer?** 
Nós criamos o núcleo de todas as IAs famosas que jogam xadrez, Dota ou empilham caixinhas (como as da OpenAI). Definimos regras (um ambiente matemático) e deixamos um computador testar todas as possibilidades até que sua Tabela de Recompensas (`Q-Table`) guardasse os movimentos ótimos para vencer o jogo sempre!